## Implementation details
1. 전처리: median fillna, bandpass filter(0.5~8Hz) + z-score normalize
2. window_size: 1m/2m의 시간 window를 기준으로 나누고, 각 window 내부 peak들로 IBI를 계산
3. BVP → peak detection → `find_peaks`
4. IBI 계산: `ibi = np.diff(peaks) / fs`
5. IBI 이상치 제거 + interpolation 후 Welch
    
    (1) IBI를 시간축 위의 신호로 만든다
    
    ```
    (1.6s에서 IBI=0.8)
    (2.4s에서 IBI=0.8)
    (3.3s에서 IBI=0.9)
    (4.1s에서 IBI=0.8)
    ```
    
    (2) 이걸 균일한 시간축으로 리샘플링한다. 중간 값들은 선형 보간으로 채운다.
    
    (3) 이렇게 균일 샘플링된 IBI 신호에 Welch 적용 → PSD → LF/HF
    
6. 각 window마다 RMSSD, SDNN, LF, HF, LF/HF 계산
7. CSV 저장: `subject | window_id | start_time | end_time | label | RMSSD | SDNN | LF | HF | LF_HF`

In [1]:
import os
import glob
import pickle
import numpy as np
import pandas as pd

from scipy.signal import butter, filtfilt, find_peaks, welch
from scipy.interpolate import interp1d


# =============================================================================
# CONFIG
# =============================================================================
PKL_DIR = r"/content/drive/MyDrive/Colab Notebooks/BP/data/WESAD/pkl_data"
OUTPUT_DIR = r"/content/drive/MyDrive/Colab Notebooks/BP/HRV-csv"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FS_LABEL = 700
FS_BVP = 64

LABEL_BASELINE = 1
LABEL_STRESS = 2
LABEL_MEDITATION = 4

STATUS_MAP = {
    LABEL_BASELINE: 0,    # baseline
    LABEL_STRESS: 1,      # stress
    LABEL_MEDITATION: 2   # meditation
}
STATUS_NAME_MAP = {
    LABEL_BASELINE: "baseline",
    LABEL_STRESS: "stress",
    LABEL_MEDITATION: "meditation"
}

LOWCUT = 0.5
HIGHCUT = 8.0
FILTER_ORDER = 4

# window configs
WINDOW_LIST_SEC = [60, 120]   # 1 min, 2 min
STEP_RATIO = 1.0              # non-overlap => step = window_sec

# label window constraints
MAJORITY_RATIO_MIN = 0.95

# peak detection
MIN_HR_BPM = 40
MAX_HR_BPM = 180
MIN_PEAK_DISTANCE_SEC = 60.0 / MAX_HR_BPM   # ~0.333 sec

# IBI filtering
IBI_MIN_SEC = 0.30
IBI_MAX_SEC = 1.50
IBI_DIFF_RATIO_MAX = 0.30   # median IBI 대비 ±30% 허용

# interpolation / PSD
FS_IBI_INTERP = 4.0         # common choice for HRV frequency analysis
LF_LOW = 0.04
LF_HIGH = 0.15
HF_LOW = 0.15
HF_HIGH = 0.40


# =============================================================================
# Basic utils
# =============================================================================
def fill_nan_with_median(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32).reshape(-1).copy()
    if np.isnan(x).any():
        med = np.nanmedian(x)
        if np.isnan(med):
            med = 0.0
        x = np.nan_to_num(x, nan=med, posinf=med, neginf=med)
    return x


def bandpass_filter(sig, fs=64, low=0.5, high=8.0, order=4):
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    nyq = 0.5 * fs
    low_n = low / nyq
    high_n = high / nyq

    if not (0 < low_n < high_n < 1):
        raise ValueError(f"Invalid bandpass range: low={low}, high={high}, fs={fs}")

    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, sig).astype(np.float32)


def zscore_1d(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    mu = float(np.mean(x))
    sd = float(np.std(x))
    if sd < 1e-8:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - mu) / (sd + 1e-8)).astype(np.float32)


def safe_div(a, b):
    return float(a / b) if abs(b) > 1e-12 else np.nan


# =============================================================================
# Find target blocks: first baseline, first stress, last meditation
# =============================================================================
def find_target_blocks(labels: np.ndarray):
    labels = np.asarray(labels).reshape(-1)
    if len(labels) == 0:
        return {}

    segments = []
    start = 0
    cur = labels[0]

    for i in range(1, len(labels)):
        if labels[i] != cur:
            segments.append((int(cur), start, i))
            start = i
            cur = labels[i]
    segments.append((int(cur), start, len(labels)))

    baseline_seg = None
    stress_seg = None
    meditation_seg = None

    for lab, s, e in segments:
        if lab == LABEL_BASELINE:
            baseline_seg = (s, e)
            break

    for lab, s, e in segments:
        if lab == LABEL_STRESS:
            stress_seg = (s, e)
            break

    for lab, s, e in reversed(segments):
        if lab == LABEL_MEDITATION:
            meditation_seg = (s, e)
            break

    out = {}
    if baseline_seg is not None:
        out[LABEL_BASELINE] = baseline_seg
    if stress_seg is not None:
        out[LABEL_STRESS] = stress_seg
    if meditation_seg is not None:
        out[LABEL_MEDITATION] = meditation_seg

    return out


def window_fully_in_block(start_l: int, end_l: int, label_major: int, target_blocks: dict) -> bool:
    if label_major not in target_blocks:
        return False
    blk_s, blk_e = target_blocks[label_major]
    return (start_l >= blk_s) and (end_l <= blk_e)


# =============================================================================
# HRV helpers
# =============================================================================
def detect_ppg_peaks(sig: np.ndarray, fs: int):
    """
    BVP peak detection.
    TPV 코드 스타일 유지:
    - minimum distance based on MAX_HR_BPM
    - adaptive prominence
    """
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    distance = max(1, int(round(MIN_PEAK_DISTANCE_SEC * fs)))
    prom = max(0.10, 0.20 * float(np.std(sig)))
    peaks, props = find_peaks(sig, distance=distance, prominence=prom)
    return peaks, props


def compute_basic_peak_info(peaks: np.ndarray, fs: int):
    peaks = np.asarray(peaks, dtype=np.int64)
    if len(peaks) < 2:
        return {
            "n_peaks": int(len(peaks)),
            "ibi_mean_sec_raw": np.nan,
            "ibi_pass_ratio": 0.0,
            "valid_ibi_count": 0,
        }

    ibi_raw = np.diff(peaks) / float(fs)
    plausible = (ibi_raw >= IBI_MIN_SEC) & (ibi_raw <= IBI_MAX_SEC)
    ibi_pass_ratio = float(np.mean(plausible)) if len(ibi_raw) > 0 else 0.0

    return {
        "n_peaks": int(len(peaks)),
        "ibi_mean_sec_raw": float(np.mean(ibi_raw)) if len(ibi_raw) > 0 else np.nan,
        "ibi_pass_ratio": ibi_pass_ratio,
        "valid_ibi_count": int(np.sum(plausible)),
    }


def build_filtered_ibi_series(peaks: np.ndarray, fs: int):
    """
    peak_times -> ibi_times / ibi
    1) plausible range filtering
    2) median 기준 ±30% 급변 제거
    """
    peaks = np.asarray(peaks, dtype=np.int64)

    if len(peaks) < 3:
        return None

    peak_times = peaks.astype(np.float64) / float(fs)   # seconds
    ibi = np.diff(peak_times)                           # seconds
    ibi_times = peak_times[1:]                          # each IBI is aligned to the later beat time

    if len(ibi) < 2:
        return None

    plausible = (ibi >= IBI_MIN_SEC) & (ibi <= IBI_MAX_SEC)
    ibi = ibi[plausible]
    ibi_times = ibi_times[plausible]

    if len(ibi) < 2:
        return None

    med_ibi = float(np.median(ibi))
    stable = np.abs(ibi - med_ibi) <= (IBI_DIFF_RATIO_MAX * (med_ibi + 1e-8))

    ibi = ibi[stable]
    ibi_times = ibi_times[stable]

    if len(ibi) < 2:
        return None

    return {
        "peak_times": peak_times,
        "ibi_times": ibi_times,
        "ibi": ibi,
        "ibi_median_sec": med_ibi,
    }


def interpolate_ibi(ibi_times: np.ndarray, ibi: np.ndarray, fs_interp: float = FS_IBI_INTERP):
    """
    Unevenly sampled IBI -> uniformly sampled IBI
    """
    ibi_times = np.asarray(ibi_times, dtype=np.float64).reshape(-1)
    ibi = np.asarray(ibi, dtype=np.float64).reshape(-1)

    if len(ibi_times) < 2 or len(ibi) < 2:
        return None

    if ibi_times[-1] <= ibi_times[0]:
        return None

    t_uniform = np.arange(ibi_times[0], ibi_times[-1], 1.0 / fs_interp)
    if len(t_uniform) < 4:
        return None

    try:
        f_interp = interp1d(
            ibi_times,
            ibi,
            kind="linear",
            bounds_error=False,
            fill_value="extrapolate",
            assume_sorted=True
        )
        ibi_uniform = f_interp(t_uniform)
    except Exception:
        return None

    ibi_uniform = np.asarray(ibi_uniform, dtype=np.float64)
    if np.any(~np.isfinite(ibi_uniform)):
        return None

    return {
        "t_uniform": t_uniform,
        "ibi_uniform": ibi_uniform,
    }


def compute_freq_domain_hrv(ibi_uniform: np.ndarray, fs_interp: float = FS_IBI_INTERP):
    """
    Welch PSD on uniformly sampled IBI
    """
    ibi_uniform = np.asarray(ibi_uniform, dtype=np.float64).reshape(-1)

    if len(ibi_uniform) < 8:
        return {
            "LF": np.nan,
            "HF": np.nan,
            "LF_HF": np.nan,
        }

    nperseg = min(256, len(ibi_uniform))
    f, pxx = welch(ibi_uniform, fs=fs_interp, nperseg=nperseg)

    lf_mask = (f >= LF_LOW) & (f < LF_HIGH)
    hf_mask = (f >= HF_LOW) & (f < HF_HIGH)

    lf = np.trapz(pxx[lf_mask], f[lf_mask]) if np.any(lf_mask) else np.nan
    hf = np.trapz(pxx[hf_mask], f[hf_mask]) if np.any(hf_mask) else np.nan
    lf_hf = safe_div(lf, hf) if np.isfinite(lf) and np.isfinite(hf) else np.nan

    return {
        "LF": float(lf) if np.isfinite(lf) else np.nan,
        "HF": float(hf) if np.isfinite(hf) else np.nan,
        "LF_HF": float(lf_hf) if np.isfinite(lf_hf) else np.nan,
    }


def compute_time_domain_hrv(ibi: np.ndarray):
    ibi = np.asarray(ibi, dtype=np.float64).reshape(-1)

    if len(ibi) < 2:
        return {
            "RMSSD": np.nan,
            "SDNN": np.nan,
            "ibi_mean_sec": np.nan,
            "ibi_std_sec": np.nan,
        }

    diff_ibi = np.diff(ibi)

    rmssd = np.sqrt(np.mean(diff_ibi ** 2)) if len(diff_ibi) > 0 else np.nan
    sdnn = np.std(ibi, ddof=0)

    return {
        "RMSSD": float(rmssd) if np.isfinite(rmssd) else np.nan,
        "SDNN": float(sdnn) if np.isfinite(sdnn) else np.nan,
        "ibi_mean_sec": float(np.mean(ibi)) if np.isfinite(np.mean(ibi)) else np.nan,
        "ibi_std_sec": float(np.std(ibi, ddof=0)) if np.isfinite(np.std(ibi, ddof=0)) else np.nan,
    }


def extract_hrv_features_from_bvp_segment(seg_bvp_raw: np.ndarray, fs_bvp: int):
    """
    Full pipeline for one BVP window:
    fillna -> bandpass -> zscore -> peaks -> filtered IBI -> interpolation -> HRV
    """
    seg_bvp = fill_nan_with_median(seg_bvp_raw)
    seg_bvp_bp = bandpass_filter(seg_bvp, fs=fs_bvp, low=LOWCUT, high=HIGHCUT, order=FILTER_ORDER)
    seg_bvp_z = zscore_1d(seg_bvp_bp)

    peaks, props = detect_ppg_peaks(seg_bvp_z, fs=fs_bvp)
    peak_info = compute_basic_peak_info(peaks, fs_bvp)

    ibi_pack = build_filtered_ibi_series(peaks, fs_bvp)

    # default row for failure cases
    row = {
        "n_peaks": peak_info["n_peaks"],
        "ibi_pass_ratio": peak_info["ibi_pass_ratio"],
        "valid_ibi_count": 0,
        "ibi_mean_sec_raw": peak_info["ibi_mean_sec_raw"],
        "ibi_mean_sec": np.nan,
        "ibi_std_sec": np.nan,
        "RMSSD": np.nan,
        "SDNN": np.nan,
        "LF": np.nan,
        "HF": np.nan,
        "LF_HF": np.nan,
    }

    if ibi_pack is None:
        return row

    ibi = ibi_pack["ibi"]
    ibi_times = ibi_pack["ibi_times"]

    row["valid_ibi_count"] = int(len(ibi))

    td = compute_time_domain_hrv(ibi)
    row.update(td)

    interp_pack = interpolate_ibi(ibi_times, ibi, fs_interp=FS_IBI_INTERP)
    if interp_pack is None:
        return row

    fd = compute_freq_domain_hrv(interp_pack["ibi_uniform"], fs_interp=FS_IBI_INTERP)
    row.update(fd)

    return row


# =============================================================================
# Build table for one subject / one window size
# =============================================================================
def build_subject_hrv_table(pkl_path: str, subject_name: str, window_sec: int) -> pd.DataFrame:
    with open(pkl_path, "rb") as f:
        s = pickle.load(f, encoding="latin1")

    bvp = np.asarray(s["signal"]["wrist"]["BVP"]).reshape(-1).astype(np.float32)
    labels = np.asarray(s["label"]).reshape(-1).astype(np.int64)

    # align label length to wrist duration
    dur_wrist = len(bvp) / FS_BVP
    labels = labels[: int(dur_wrist * FS_LABEL)]

    bvp = fill_nan_with_median(bvp)

    target_blocks = find_target_blocks(labels)
    if len(target_blocks) == 0:
        return pd.DataFrame()

    Wl = int(window_sec * FS_LABEL)
    Sl = int(window_sec * STEP_RATIO * FS_LABEL)
    Wb = int(window_sec * FS_BVP)

    rows = []
    window_counter = 0

    for start_l in range(0, len(labels) - Wl + 1, Sl):
        end_l = start_l + Wl
        win_lab = labels[start_l:end_l]

        binc = np.bincount(win_lab)
        maj = int(np.argmax(binc))
        maj_ratio = float((win_lab == maj).mean())

        if maj not in [LABEL_BASELINE, LABEL_STRESS, LABEL_MEDITATION]:
            continue
        if maj_ratio < MAJORITY_RATIO_MIN:
            continue
        if not window_fully_in_block(start_l, end_l, maj, target_blocks):
            continue

        t0 = start_l / FS_LABEL
        t1 = t0 + window_sec

        start_b = int(round(t0 * FS_BVP))
        end_b = start_b + Wb

        if end_b > len(bvp):
            continue

        seg_bvp_raw = bvp[start_b:end_b]
        if len(seg_bvp_raw) != Wb:
            continue

        hrv_info = extract_hrv_features_from_bvp_segment(seg_bvp_raw, FS_BVP)

        window_counter += 1

        row = {
            "subject": subject_name,
            "window": f"window{window_counter}",
            "window_index": window_counter,
            "window_sec": int(window_sec),
            "status": STATUS_MAP[maj],
            "status_name": STATUS_NAME_MAP[maj],
            "label_major": maj,
            "t_start_sec": float(t0),
            "t_end_sec": float(t1),
            "major_ratio": maj_ratio,
        }

        for k, v in hrv_info.items():
            row[k] = v

        rows.append(row)

    return pd.DataFrame(rows)


# =============================================================================
# Build all subjects for one window size
# =============================================================================
def build_all_subjects_hrv_table(pkl_dir: str, window_sec: int) -> pd.DataFrame:
    pkl_list = sorted(glob.glob(os.path.join(pkl_dir, "S*.pkl")))
    all_dfs = []

    print(f"[INFO] Found {len(pkl_list)} pkl files")
    print(f"[INFO] Building HRV table for window_sec={window_sec}")

    for pkl_path in pkl_list:
        subject_name = os.path.splitext(os.path.basename(pkl_path))[0]
        print(f"\n[INFO] Processing {subject_name} ...")
        try:
            df_sub = build_subject_hrv_table(pkl_path, subject_name, window_sec=window_sec)
            print(f"[INFO] {subject_name}: extracted {len(df_sub)} target windows")
            if len(df_sub) > 0:
                all_dfs.append(df_sub)
        except Exception as e:
            print(f"[WARN] Failed on {subject_name}: {e}")

    if len(all_dfs) == 0:
        return pd.DataFrame()

    return pd.concat(all_dfs, axis=0, ignore_index=True)


# =============================================================================
# Save helpers
# =============================================================================
def save_hrv_outputs(df_all: pd.DataFrame, window_sec: int, output_dir: str):
    if len(df_all) == 0:
        print(f"[WARN] No windows extracted for {window_sec}s")
        return

    base_cols = [
        "subject", "window", "window_index", "window_sec",
        "status", "status_name", "label_major",
        "t_start_sec", "t_end_sec", "major_ratio"
    ]

    hrv_cols = [
        "n_peaks",
        "ibi_pass_ratio",
        "valid_ibi_count",
        "ibi_mean_sec_raw",
        "ibi_mean_sec",
        "ibi_std_sec",
        "RMSSD",
        "SDNN",
        "LF",
        "HF",
        "LF_HF",
    ]

    final_cols = base_cols + hrv_cols
    df_all = df_all[final_cols].copy()

    minute_tag = f"{window_sec // 60}min"

    csv_path = os.path.join(output_dir, f"HRV_{minute_tag}.csv")
    df_all.to_csv(csv_path, index=False)
    print(f"[INFO] Saved: {csv_path}")

    summary = (
        df_all.groupby(["subject", "status_name"])
        .agg(
            n_windows=("window", "count"),
            n_valid_rmssd=("RMSSD", lambda x: int(np.sum(np.isfinite(x)))),
            n_valid_lf_hf=("LF_HF", lambda x: int(np.sum(np.isfinite(x)))),
        )
        .reset_index()
    )

    summary_csv = os.path.join(output_dir, f"HRV_{minute_tag}_summary.csv")
    summary.to_csv(summary_csv, index=False)
    print(f"[INFO] Saved: {summary_csv}")

    print(f"\n[INFO] Status counts ({minute_tag})")
    print(df_all["status_name"].value_counts())

    print(f"\n[INFO] Summary ({minute_tag})")
    print(summary.to_string(index=False))


# =============================================================================
# Main
# =============================================================================
if __name__ == "__main__":
    for window_sec in WINDOW_LIST_SEC:
        df_all = build_all_subjects_hrv_table(PKL_DIR, window_sec=window_sec)

        print(f"\n[INFO] final raw shape for {window_sec}s:", df_all.shape)
        if len(df_all) == 0:
            print(f"[WARN] No target windows extracted for {window_sec}s")
            continue

        save_hrv_outputs(df_all, window_sec=window_sec, output_dir=OUTPUT_DIR)

[INFO] Found 15 pkl files
[INFO] Building HRV table for window_sec=60

[INFO] Processing S10 ...


/tmp/ipykernel_502/51286544.py:300: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  lf = np.trapz(pxx[lf_mask], f[lf_mask]) if np.any(lf_mask) else np.nan
/tmp/ipykernel_502/51286544.py:301: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  hf = np.trapz(pxx[hf_mask], f[hf_mask]) if np.any(hf_mask) else np.nan


[INFO] S10: extracted 36 target windows

[INFO] Processing S11 ...
[INFO] S11: extracted 35 target windows

[INFO] Processing S13 ...
[INFO] S13: extracted 35 target windows

[INFO] Processing S14 ...
[INFO] S14: extracted 35 target windows

[INFO] Processing S15 ...
[INFO] S15: extracted 35 target windows

[INFO] Processing S16 ...
[INFO] S16: extracted 35 target windows

[INFO] Processing S17 ...
[INFO] S17: extracted 36 target windows

[INFO] Processing S2 ...
[INFO] S2: extracted 33 target windows

[INFO] Processing S3 ...
[INFO] S3: extracted 34 target windows

[INFO] Processing S4 ...
[INFO] S4: extracted 34 target windows

[INFO] Processing S5 ...
[INFO] S5: extracted 33 target windows

[INFO] Processing S6 ...
[INFO] S6: extracted 34 target windows

[INFO] Processing S7 ...
[INFO] S7: extracted 35 target windows

[INFO] Processing S8 ...
[INFO] S8: extracted 36 target windows

[INFO] Processing S9 ...
[INFO] S9: extracted 32 target windows

[INFO] final raw shape for 60s: (518,

/tmp/ipykernel_502/51286544.py:300: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  lf = np.trapz(pxx[lf_mask], f[lf_mask]) if np.any(lf_mask) else np.nan
/tmp/ipykernel_502/51286544.py:301: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  hf = np.trapz(pxx[hf_mask], f[hf_mask]) if np.any(hf_mask) else np.nan


[INFO] S10: extracted 17 target windows

[INFO] Processing S11 ...
[INFO] S11: extracted 16 target windows

[INFO] Processing S13 ...
[INFO] S13: extracted 15 target windows

[INFO] Processing S14 ...
[INFO] S14: extracted 16 target windows

[INFO] Processing S15 ...
[INFO] S15: extracted 16 target windows

[INFO] Processing S16 ...
[INFO] S16: extracted 17 target windows

[INFO] Processing S17 ...
[INFO] S17: extracted 17 target windows

[INFO] Processing S2 ...
[INFO] S2: extracted 16 target windows

[INFO] Processing S3 ...
[INFO] S3: extracted 16 target windows

[INFO] Processing S4 ...
[INFO] S4: extracted 15 target windows

[INFO] Processing S5 ...
[INFO] S5: extracted 15 target windows

[INFO] Processing S6 ...
[INFO] S6: extracted 15 target windows

[INFO] Processing S7 ...
[INFO] S7: extracted 17 target windows

[INFO] Processing S8 ...
[INFO] S8: extracted 17 target windows

[INFO] Processing S9 ...
[INFO] S9: extracted 15 target windows

[INFO] final raw shape for 120s: (240